# Google Colab Example for eDensGPU_MP2

This notebook runs an example MP2 correlation energy density evaluation on Google Colab. It showcases the GPU-based implementation that uses the auxiliary basis expansion. 

## Table of contents 
- [Installation](#1-installation)
- [Molecule Setup](#2-molecule-setup)
- [Run `ec_mp2_gen`](#3-run-ec_mp2_gen)

## 1) Installation
For the Google Colab environment setup, select first a suitable runtime:

**Runtime → Change runtime type → Runtime Version → 2026.07**, with
any **NVIDIA GPU**. 

This runtime allows **Python 3.12** (see
[Colab's runtime list](https://research.google.com/colaboratory/runtime-version-faq.html)).

Installation uses the latest commit on the public GitHub `main` branch. Set `REVISION` to a release tag to install a fixed version instead. 

Normal pip installs **CuPy, GPU4PySCF, cuTENSOR** and the pinned numerical libraries. Run this cell first, before importing scientific packages. Continue directly to section 2 when installation succeeds. This package accepts NumPy 1.26.4 and NumPy 2.0–2.2, matching the pinned CuPy support range. A compatible NumPy already installed in Colab can be retained. If an already-loaded package differs from the installed version, restart the session once and continue with section 2. The installer checks for this condition; rerunning pip cannot replace compiled libraries already loaded in memory.

In [ ]:
import importlib
import importlib.metadata as metadata
import os
import subprocess
import sys

REPOSITORY = "https://github.com/vuckovic-lab/eDensGPU_MP2.git"
REVISION = "main"  # Replace with a release tag to install a specific version; "main" installs the latest commit. 
 
if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Select Colab runtime 2026.07 (Python 3.12) with an NVIDIA GPU.")
print("Checking for an NVIDIA GPU ...")
subprocess.run(["nvidia-smi"], check=True)
print(f"Installing eDensGPU_MP2 from {REVISION} ...")
# Record packages already loaded by the notebook kernel.
loaded_roots = {name.split(".", 1)[0] for name in tuple(sys.modules)}
loaded_distributions = {
    distribution
    for module, distributions in metadata.packages_distributions().items()
    if module in loaded_roots
    for distribution in distributions
}
loaded_versions = {name: metadata.version(name) for name in loaded_distributions}

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-input",
     f"git+{REPOSITORY}@{REVISION}"],
    check=True,
    env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
)
importlib.invalidate_caches()
print(f"Installation complete with NumPy {metadata.version('numpy')}. Continue directly with section 2.")


## 2) Molecule setup

Choose your molecule system and its parameters for the PySCF evaluation. 
Note that PySCF uses `spin=2S`. Set scf type to `rhf` for closed-shell system or `uhf` for open-shell system. 



In [ ]:
import time
import cupy as cp
from pyscf import gto
from gpu4pyscf import dft

if cp.cuda.runtime.getDeviceCount() < 1:
    raise RuntimeError("Select an NVIDIA GPU runtime.")

###==== Define molecule and basis set ====###
struc = """
O   0.000000   0.000000   0.000000
H   0.000000  -0.757000   0.587000
H   0.000000   0.757000   0.587000
"""
charge = 0
spin = 0
basis = "def2qzvppd"
scf_type = "rhf"       # "rhf" or "uhf"
xc_fun = 'hf'


###==== Run SCF calculation ==###
mol = gto.M(atom=struc, basis=basis, charge=charge, spin=spin,
            unit="Angstrom", cart=False, verbose=0)
if scf_type == "rhf":
    mf = dft.RKS(mol)
    mf.xc = xc_fun
elif scf_type == "uhf":
    mf = dft.UKS(mol)
    mf.xc = xc_fun
cp.cuda.runtime.deviceSynchronize()
scf_started = time.perf_counter()
mf.kernel()
cp.cuda.runtime.deviceSynchronize()
scf_seconds = time.perf_counter() - scf_started
print("HF energy / Hartree:", repr(float(mf.e_tot)))
print(f"SCF elapsed: {scf_seconds:.3f} s")


## 3) Run `eDensGPU_MP2`
MP2 correlation energy density generation uses regularization `kappa=2.0` as the default. Choose the grid level and fitting basis independently of the orbital basis.

The returned `ec_out` object contains `tot` and `os` correlation energies in Hartree, `dens_tot` and `dens_os` energy densities per particle, and `rho`, `weights` and `coords` as float64 CuPy arrays.


In [ ]:
import time
import cupy as cp
from eDens_mp2 import ec_mp2_gen, make_grid

###==== eDensGPU_MP2 parameters ====###
grid_level = 3
kappa = 2.0
auxbasis = basis + "-ri"
coords, weights = make_grid(mol, level=grid_level)

###==== Run eDensGPU_MP2 calculation ====###
cp.cuda.runtime.deviceSynchronize()
mp2_started = time.perf_counter()
ec_out = ec_mp2_gen(mf, mol, coords, weights,
                     kappa=kappa, auxbasis=auxbasis, verbose=True)
cp.cuda.runtime.deviceSynchronize()
mp2_seconds = time.perf_counter() - mp2_started
print(f"Generation elapsed: {mp2_seconds:.3f} s")

print("Correlation energies / Hartree:")
print("Total:", repr(ec_out.tot))
print("OS:   ", repr(ec_out.os))
